# dbt Cloud Debugging & Code Review

## Purpose

This guide is structured for a hands-on technical interview (60 minutes) focused on debugging dbt Cloud projects. You will learn how to systematically diagnose and fix errors in dbt models, tests, and configurations. This guide covers the full error spectrum: YAML parsing failures, Jinja compilation errors, SQL runtime errors, test failures, and DAG/dependency issues. The goal is to develop the debugging mindset and techniques that data engineers use daily.

## Table of Contents

1. [How to Read dbt Error Output](#section-1)
2. [Error Classification Decision Tree](#section-2)
3. [Pattern Library](#section-3)
   - [Pattern A: YAML & Configuration Errors](#pattern-a)
   - [Pattern B: Compilation Errors (Jinja/ref/source)](#pattern-b)
   - [Pattern C: Database/SQL Runtime Errors](#pattern-c)
   - [Pattern D: Test Failures](#pattern-d)
   - [Pattern E: Dependency & DAG Errors](#pattern-e)
   - [Pattern F: Incremental Model Issues](#pattern-f)
   - [Pattern G: Source Freshness & Environment Issues](#pattern-g)
4. [Code Review Checklist for dbt](#section-4)
5. [Debugging Workflow in dbt Cloud](#section-5)
6. [Common Interview Traps](#section-6)


<hr style="border: 3px solid black;">

# Section 1: How to Read dbt Error Output {#section-1}

## Anatomy of a dbt Cloud Error

dbt errors occur at multiple layers. Understanding which layer failed is the first step to fixing the problem.

### The dbt Error Hierarchy

Errors propagate upward through this stack:

1. **YAML Parsing** → dbt reads schema.yml and models (must be valid YAML)
2. **Jinja Compilation** → dbt renders Jinja templates and resolves ref()/source() calls
3. **SQL Compilation** → dbt generates final SQL and sends to warehouse
4. **Database Execution** → warehouse executes SQL and may return SQL errors
5. **Test Execution** → dbt runs tests against the model data

If a model fails at step 1 (YAML), it never reaches step 2. If it fails at step 2 (Jinja), the compiled SQL is never generated.

### Real Example: Dissecting a dbt Error

```
ERROR in model my_first_dbt_model (models/example/my_first_dbt_model.sql):
  Compilation Error in model my_first_dbt_model (models/example/my_first_dbt_model.sql)
    ('str' object does not support item assignment)
  error in macro render (macros/generate_alias_explore.sql): 'str' object does not support item assignment
```

**Labeled Parts:**
- **ERROR in model**: The file that failed (models/example/my_first_dbt_model.sql)
- **Compilation Error**: This is a Jinja/macro error during compilation, not a database error
- **Error message**: 'str' object does not support item assignment (indicates bad Jinja syntax)
- **Macro name**: error in macro render (tells you the exact macro where the error occurred)

### Types of dbt Errors

| Error Level | What It Means | Where to Look | Example |
|---|---|---|---|
| **YAML Parsing Error** | Invalid YAML syntax in schema.yml or dbt_project.yml | dbt logs show "Error reading model definition" | tabs instead of spaces, wrong nesting |
| **Compilation Error** | Jinja template error, undefined ref/source, macro issue | dbt compile output, check Jinja logic and macro calls | `ref('nonexistent_model')`, missing closing `%}` |
| **SQL Runtime Error** | Database rejected the SQL (invalid syntax, missing column, permission denied) | dbt Cloud Logs panel, warehouse query history | Column does not exist, type mismatch, division by zero |
| **Test Failure** | Data in model violates test assertion | dbt test output, shows which rows failed | unique constraint violated, NULL values found |
| **DAG/Dependency Error** | Circular reference or unresolved dependency | dbt parse output, use `dbt DAG` to inspect | model A refs B, B refs A |

### Interview Tip: dbt-Level vs Warehouse-Level Errors

**dbt-level errors** stop the process before SQL is sent to the warehouse (YAML parsing, Jinja compilation, ref/source resolution). Look at dbt logs.

**Warehouse-level errors** occur during SQL execution or test evaluation. The SQL is valid dbt SQL but the warehouse rejects it. Check warehouse query history and error messages.

To distinguish: if dbt Cloud shows a red error in the Logs panel BEFORE it mentions the warehouse, it's a dbt-level error. If it shows "executing SQL" and then fails, it's a warehouse error.


<hr style="border: 3px solid black;">

# Section 2: Error Classification Decision Tree {#section-2}

When you see an error, use this flowchart to classify it:

<div class="fc">
  <div class="fc-node fc-start">dbt Run Failed — Where did it fail?</div>
  <div class="fc-arrow">▼</div>
  <div class="fc-branches">
    <div class="fc-branch">
      <div class="fc-label fc-tag">BEFORE SQL SENT</div>
      <div class="fc-node fc-action">Error logged before SQL reached the warehouse</div>
      <div class="fc-arrow">▼</div>
      <div class="fc-branches fc-three">
        <div class="fc-branch">
          <div class="fc-label fc-tag">YAML</div>
          <div class="fc-node fc-warn"><strong>Pattern A</strong><br/>Indentation, missing fields, duplicates</div>
        </div>
        <div class="fc-branch">
          <div class="fc-label fc-tag">JINJA</div>
          <div class="fc-node fc-warn"><strong>Pattern B</strong><br/>Jinja syntax, macros, variables</div>
        </div>
        <div class="fc-branch">
          <div class="fc-label fc-tag">REF / SOURCE</div>
          <div class="fc-node fc-warn"><strong>Pattern B</strong><br/>Missing ref(), undefined source, circular deps</div>
        </div>
      </div>
    </div>
    <div class="fc-branch">
      <div class="fc-label fc-tag">DURING SQL EXECUTION</div>
      <div class="fc-node fc-warn"><strong>Pattern C</strong><br/>Database/SQL runtime error<br/>Column not found, type mismatch, permissions</div>
    </div>
  </div>
</div>

<div class="fc">
  <div class="fc-node fc-start">Tests Failed?</div>
  <div class="fc-arrow">▼</div>
  <div class="fc-branches">
    <div class="fc-branch">
      <div class="fc-label fc-yes">YES</div>
      <div class="fc-node fc-warn"><strong>Pattern D</strong> — Test Failure<br/>unique, not_null, accepted_values, relationships<br/>Data issue, not code issue — trace upstream</div>
    </div>
    <div class="fc-branch">
      <div class="fc-label fc-no">NO</div>
      <div class="fc-node fc-good">Model compiled and ran successfully</div>
    </div>
  </div>
</div>

<div class="fc">
  <div class="fc-node fc-start">Incremental model issues?</div>
  <div class="fc-arrow">▼</div>
  <div class="fc-branches">
    <div class="fc-branch">
      <div class="fc-label fc-tag">DUPLICATES</div>
      <div class="fc-node fc-warn"><strong>Pattern F1</strong><br/>Missing or wrong unique_key</div>
    </div>
    <div class="fc-branch">
      <div class="fc-label fc-tag">SCHEMA CHANGE</div>
      <div class="fc-node fc-warn"><strong>Pattern F2</strong><br/>New column not added<br/>Use --full-refresh</div>
    </div>
    <div class="fc-branch">
      <div class="fc-label fc-tag">LOGIC ERROR</div>
      <div class="fc-node fc-warn"><strong>Pattern F3</strong><br/>is_incremental() behaves differently<br/>on full vs incremental run</div>
    </div>
  </div>
</div>

<div class="fc">
  <div class="fc-node fc-start">Freshness / Environment issues?</div>
  <div class="fc-arrow">▼</div>
  <div class="fc-branches">
    <div class="fc-branch">
      <div class="fc-label fc-tag">STALE DATA</div>
      <div class="fc-node fc-warn"><strong>Pattern G</strong><br/>Source freshness check failed</div>
    </div>
    <div class="fc-branch">
      <div class="fc-label fc-tag">ENV / PROFILE</div>
      <div class="fc-node fc-warn"><strong>Pattern G</strong><br/>Missing env var, wrong target, package conflict</div>
    </div>
  </div>
</div>

### Key Insight: dbt Errors Bubble Up

The error hierarchy is **sequential**. dbt always checks YAML first, then Jinja, then SQL. If you see a Jinja error, YAML is valid. If you see a SQL error, YAML and Jinja are valid.

Example flow:
- You fix indentation in schema.yml (YAML) ✓
- dbt now parses YAML successfully ✓
- dbt tries to compile Jinja and hits `ref('nonexistent_model')` ✗ (Pattern B)
- You fix the ref() name ✓
- dbt compiles and sends SQL to warehouse ✓
- Warehouse executes and hits "column not found" ✗ (Pattern C)
- You update the column name in the SQL ✓
- dbt runs successfully ✓
- dbt test runs and fails on data validation ✗ (Pattern D)
- You trace the upstream data issue and fix the root cause ✓

<hr style="border: 3px solid black;">

# Section 3: Pattern Library {#section-3}

This section contains the most common dbt debugging patterns. Each pattern shows the error message, the broken code, the explanation, and the fix.

Use this as a reference during the interview. When you encounter an error, match it to a pattern and apply the fix.


<hr style="border: 2px solid black;">

## Pattern A: YAML & Configuration Errors {#pattern-a}

### A1: Indentation Error (Tabs vs Spaces)

**Error Message:**
```
Invalid yaml structure: expected <block end>, but found '<block mapping start>'
```

**Broken YAML (models/schema.yml):**
```yaml
version: 2
models:
  - name: my_model
    columns:
    - name: id  # Used tab instead of 4 spaces!
      data_type: integer
```

**Explanation:**
dbt requires YAML indentation to be consistent (typically 2 or 4 spaces). If you mix tabs and spaces, or use inconsistent spacing, the YAML parser fails.

**Fix:**
```yaml
version: 2
models:
  - name: my_model
    columns:
      - name: id
        data_type: integer
```
Use spaces only, 2 or 4 consistently. Most editors have a setting to show/replace tabs.

---

### A2: Missing Required Field

**Error Message:**
```
Error in model definition (models/schema.yml): Missing required field 'name'
```

**Broken YAML (models/schema.yml):**
```yaml
version: 2
models:
  - columns:  # Missing 'name' field!
      - name: id
```

**Explanation:**
Every model definition must have a `name` field. Same for sources, columns, and tests.

**Fix:**
```yaml
version: 2
models:
  - name: my_model
    columns:
      - name: id
```

---

### A3: Duplicate Model/Source Names

**Error Message:**
```
Error: dbt found two definitions of model 'my_model'
```

**Broken YAML (models/schema.yml):**
```yaml
version: 2
models:
  - name: my_model
    description: First definition
  - name: my_model  # Duplicate!
    description: Second definition
```

**Explanation:**
dbt model names must be unique within a project. If you define the same model twice (perhaps across two schema.yml files), dbt will error.

**Fix:**
Rename one of the duplicates or merge their definitions into a single model block:
```yaml
version: 2
models:
  - name: my_model
    description: Unified description
    columns:
      - name: id
```

---

### A4: Invalid Test Configuration

**Error Message:**
```
Error: Test 'invalid_test_name' does not exist
```

**Broken YAML (models/schema.yml):**
```yaml
version: 2
models:
  - name: my_model
    columns:
      - name: id
        tests:
          - invalid_test_name  # Typo!
          - unique
```

**Explanation:**
You referenced a test that doesn't exist (either typo in name or test not in project). dbt has built-in tests: `unique`, `not_null`, `accepted_values`, `relationships`.

**Fix:**
```yaml
version: 2
models:
  - name: my_model
    columns:
      - name: id
        tests:
          - unique
          - not_null
```
Or if using a custom test, ensure the test SQL file exists in `tests/` and the test name matches the filename.


<hr style="border: 2px solid black;">

## Pattern B: Compilation Errors (Jinja/ref/source) {#pattern-b}

### B1: ref() to Non-Existent Model

**Error Message:**
```
ERROR: 'nonexistent_model' does not exist in this dbt project!
```

**Broken Code (models/marts/fact_orders.sql):**
```sql
SELECT
  o.order_id,
  c.customer_name
FROM {{ ref('nonexistent_model') }} o  -- Typo in model name
JOIN {{ ref('customers') }} c
  ON o.customer_id = c.customer_id
```

**Explanation:**
You referenced a model in ref() that doesn't exist. Common causes: typo, model deleted, model in wrong folder.

**Fix:**
1. Check the exact filename of the model in your project.
2. Fix the ref() to match:
```sql
FROM {{ ref('stg_orders') }} o  -- Correct model name
```

---

### B2: source() Referencing Undefined Source

**Error Message:**
```
ERROR: source 'raw.nonexistent_table' does not exist in this dbt project!
```

**Broken Code (models/staging/stg_orders.sql):**
```sql
SELECT
  order_id,
  customer_id
FROM {{ source('raw', 'nonexistent_table') }}  -- Typo in source name
```

**Explanation:**
You referenced a source that is not defined in schema.yml. The source definition must exist and match the source_name and table_name.

**Fix:**
1. Check models/schema.yml (or sources.yml) for the correct source definition.
2. Update the source() call:
```sql
FROM {{ source('raw_data', 'orders') }}  -- Correct source and table name
```
Or add the missing source to schema.yml:
```yaml
version: 2
sources:
  - name: raw
    tables:
      - name: nonexistent_table
```

---

### B3: Jinja Syntax Error

**Error Message:**
```
ERROR: unexpected '}'
```

**Broken Code (models/marts/fact_orders.sql):**
```sql
SELECT
  order_id,
  {{ 'amount' | upper }  -- Missing closing }}
FROM {{ ref('orders') }}
```

**Explanation:**
Jinja uses `{{ }}` for variable interpolation and `{% %}` for control flow. Missing closing delimiters or mismatched brackets cause parse errors.

**Fix:**
```sql
SELECT
  order_id,
  {{ 'amount' | upper }}  -- Added closing }}
FROM {{ ref('orders') }}
```

---

### B4: Macro Not Found or Wrong Arguments

**Error Message:**
```
ERROR: macro 'generate_alias' does not exist
```

**Broken Code (models/staging/stg_orders.sql):**
```sql
{{ config(alias = generate_alias(customer_id)) }}  -- Macro typo or missing

SELECT * FROM {{ ref('raw_orders') }}
```

**Explanation:**
You called a macro that doesn't exist or passed wrong number of arguments. Check macros/ folder.

**Fix:**
1. Verify the macro exists in macros/ (e.g., macros/generate_alias.sql).
2. Check the macro signature for required arguments:
```sql
-- macros/generate_alias.sql
{% macro generate_alias(column_name) %}
  alias_{{ column_name }}
{% endmacro %}
```
3. Call with correct arguments:
```sql
{{ config(alias = generate_alias('customer_id')) }}
```

---

### B5: Circular Reference

**Error Message:**
```
ERROR: Circular dependency detected: model_a -> model_b -> model_a
```

**Broken Code:**
```sql
-- models/staging/model_a.sql
SELECT * FROM {{ ref('model_b') }}

-- models/staging/model_b.sql
SELECT * FROM {{ ref('model_a') }}  -- Circular!
```

**Explanation:**
Model A depends on Model B, and Model B depends on Model A. dbt cannot resolve this DAG.

**Fix:**
Break the cycle by removing one dependency. Typically, introduce an intermediate model or refactor the logic:
```sql
-- models/staging/model_a.sql
SELECT * FROM raw_table_a

-- models/staging/model_b.sql
SELECT * FROM {{ ref('model_a') }}  -- Only depends on A, not A on B
```


<hr style="border: 2px solid black;">

## Pattern C: Database/SQL Runtime Errors {#pattern-c}

### C1: Column Does Not Exist

**Error Message:**
```
ERROR: column "customer_id" does not exist
```

**Broken Code (models/marts/fact_orders.sql):**
```sql
SELECT
  order_id,
  customer_id,  -- Column doesn't exist in upstream model
  order_amount
FROM {{ ref('stg_orders') }}
```

**Explanation:**
The upstream model (stg_orders) was changed and the column name is different or was removed. Always check the upstream model schema when this occurs.

**Fix:**
1. Check the upstream model to see what columns it produces:
   - Run `dbt compile` and check the compiled SQL for stg_orders
   - Check the dbt Cloud Lineage view
2. Update the column name to match:
```sql
SELECT
  order_id,
  cust_id,  -- Correct column name from upstream
  order_amount
FROM {{ ref('stg_orders') }}
```

---

### C2: Type Mismatch in UNION or JOIN

**Error Message:**
```
ERROR: UNION query must have the same number of columns and compatible types
```

**Broken Code (models/marts/combined_orders.sql):**
```sql
SELECT
  order_id,
  order_date,  -- DATE type
  amount       -- DECIMAL type
FROM {{ ref('orders_2023') }}

UNION ALL

SELECT
  order_id,
  order_date,
  amount::VARCHAR  -- VARCHAR type - type mismatch!
FROM {{ ref('orders_2024') }}
```

**Explanation:**
UNION and JOIN require columns to have compatible data types. If amount is DECIMAL in one table and VARCHAR in another, they don't match.

**Fix:**
Cast both sides to the same type:
```sql
SELECT
  order_id,
  order_date,
  amount::DECIMAL  -- Explicit cast
FROM {{ ref('orders_2023') }}

UNION ALL

SELECT
  order_id,
  order_date,
  amount::DECIMAL  -- Match the type
FROM {{ ref('orders_2024') }}
```

---

### C3: Division by Zero

**Error Message:**
```
ERROR: division by zero
```

**Broken Code (models/marts/revenue_metrics.sql):**
```sql
SELECT
  customer_id,
  total_revenue / order_count AS avg_revenue  -- order_count can be 0!
FROM {{ ref('customer_stats') }}
```

**Explanation:**
If a denominator is zero, division fails. This often happens with aggregations where some groups have no data.

**Fix:**
Use NULLIF to prevent division by zero:
```sql
SELECT
  customer_id,
  total_revenue / NULLIF(order_count, 0) AS avg_revenue
FROM {{ ref('customer_stats') }}
```
This returns NULL instead of an error when order_count is 0.

---

### C4: Ambiguous Column Reference

**Error Message:**
```
ERROR: column reference "customer_id" is ambiguous
```

**Broken Code (models/marts/fact_orders.sql):**
```sql
SELECT
  o.order_id,
  customer_id,  -- Both tables have customer_id!
  c.customer_name
FROM {{ ref('orders') }} o
JOIN {{ ref('customers') }} c
  ON o.customer_id = c.customer_id
```

**Explanation:**
When you SELECT a column that exists in multiple joined tables without specifying which table it comes from, SQL doesn't know which one to use.

**Fix:**
Qualify the column with the table alias:
```sql
SELECT
  o.order_id,
  o.customer_id,  -- Qualified with o.
  c.customer_name
FROM {{ ref('orders') }} o
JOIN {{ ref('customers') }} c
  ON o.customer_id = c.customer_id
```

---

### C5: Permission Denied / Schema Not Found

**Error Message:**
```
ERROR: relation "raw.orders" does not exist
```

**Broken Code (models/staging/stg_orders.sql):**
```sql
SELECT * FROM raw.orders  -- Schema doesn't exist or no permission
```

**Explanation:**
Either the schema doesn't exist in the warehouse, or the dbt Cloud profile doesn't have permission to access it. This is also a sign that a raw table reference was used instead of source().

**Fix:**
1. Use source() instead of hardcoding schema.table:
```sql
SELECT * FROM {{ source('raw', 'orders') }}
```
2. Verify the source is defined in schema.yml:
```yaml
sources:
  - name: raw
    schema: raw_schema  # Adjust schema name if needed
    tables:
      - name: orders
```
3. Check dbt Cloud profile settings for the correct database/schema.


<hr style="border: 2px solid black;">

## Pattern D: Test Failures {#pattern-d}

### Key Concept: Test Failures = Data Problems

When a test fails, your **model code is valid** but your **data violates the test assertion**. This means the root cause is upstream data, not the current model. Always trace upstream to find where the bad data originates.

---

### D1: Unique Test Failure

**Error Message:**
```
Assertion failed. Expected 0, got 5 (rows where id appeared more than once)
```

**Test Definition (models/schema.yml):**
```yaml
version: 2
models:
  - name: fact_orders
    columns:
      - name: order_id
        tests:
          - unique  # Failed: order_id has duplicates
```

**What It Means:**
The column should have only unique values, but it has duplicates. This is a data quality issue.

**How to Diagnose:**
```sql
-- Find duplicate rows
SELECT
  order_id,
  COUNT(*) AS count
FROM {{ ref('fact_orders') }}
GROUP BY order_id
HAVING COUNT(*) > 1
ORDER BY count DESC;
```

**How to Fix the Root Cause:**
1. Trace upstream: where do the order_id values come from?
2. Check the source table (e.g., raw.orders) — does it have duplicates?
3. Check the staging model (stg_orders) — did it dedup correctly?
4. If duplicates are intentional (e.g., multiple items per order), use a composite key:
```yaml
- name: order_id
  tests: []
- name: line_item_id
  tests:
    - unique  # (order_id, line_item_id) together should be unique
```

---

### D2: not_null Test Failure

**Error Message:**
```
Assertion failed. Expected 0 rows with NULL, got 12.
```

**Test Definition (models/schema.yml):**
```yaml
version: 2
models:
  - name: fact_orders
    columns:
      - name: customer_id
        tests:
          - not_null  # Failed: customer_id has NULLs
```

**What It Means:**
The column should not contain NULL values, but it does. Required data is missing.

**How to Diagnose:**
```sql
SELECT COUNT(*) AS null_count
FROM {{ ref('fact_orders') }}
WHERE customer_id IS NULL;
```

**How to Fix the Root Cause:**
1. Trace upstream: does the source table have NULLs in customer_id?
2. Check the JOIN logic — are you left joining when should you inner join?
3. Add a filter to exclude NULLs in the staging model:
```sql
SELECT * FROM {{ ref('raw_orders') }}
WHERE customer_id IS NOT NULL
```
4. Or coalesce the column to a default value:
```sql
SELECT
  order_id,
  COALESCE(customer_id, -1) AS customer_id
FROM {{ ref('raw_orders') }}
```

---

### D3: accepted_values Test Failure

**Error Message:**
```
Assertion failed. Expected 0 rows with unexpected values, got 8 rows with invalid status values.
```

**Test Definition (models/schema.yml):**
```yaml
version: 2
models:
  - name: fact_orders
    columns:
      - name: order_status
        tests:
          - accepted_values:
              values: ['pending', 'completed', 'cancelled']
```

**What It Means:**
The column should only contain one of the specified values, but it has unexpected values (e.g., 'archived', 'unknown').

**How to Diagnose:**
```sql
SELECT
  order_status,
  COUNT(*) AS count
FROM {{ ref('fact_orders') }}
WHERE order_status NOT IN ('pending', 'completed', 'cancelled')
GROUP BY order_status;
```

**How to Fix the Root Cause:**
1. Decide if the new values (e.g., 'archived') are valid:
   - If yes, add them to the accepted_values list.
   - If no, trace where they come from and clean upstream.
2. Update the test:
```yaml
- accepted_values:
    values: ['pending', 'completed', 'cancelled', 'archived']
```
3. Or clean the data in the model:
```sql
SELECT
  order_id,
  CASE 
    WHEN order_status NOT IN ('pending', 'completed', 'cancelled')
    THEN 'cancelled'
    ELSE order_status
  END AS order_status
FROM {{ ref('raw_orders') }}
```

---

### D4: relationships Test Failure (Foreign Key Violation)

**Error Message:**
```
Assertion failed. Expected 0 rows with NULL or mismatched foreign key, got 23.
```

**Test Definition (models/schema.yml):**
```yaml
version: 2
models:
  - name: fact_orders
    columns:
      - name: customer_id
        tests:
          - relationships:
              to: ref('dim_customers')
              field: customer_id
```

**What It Means:**
A customer_id in fact_orders does not exist in dim_customers. The foreign key is broken.

**How to Diagnose:**
```sql
SELECT
  f.customer_id,
  COUNT(*) AS order_count
FROM {{ ref('fact_orders') }} f
LEFT JOIN {{ ref('dim_customers') }} c ON f.customer_id = c.customer_id
WHERE c.customer_id IS NULL
GROUP BY f.customer_id;
```

**How to Fix the Root Cause:**
1. Check if these customers should exist:
   - Are they recently deleted (archival issue)?
   - Are they new and not yet loaded to dim_customers?
2. Use an INNER JOIN in fact_orders to exclude them:
```sql
SELECT
  o.order_id,
  o.customer_id
FROM {{ ref('raw_orders') }} o
INNER JOIN {{ ref('dim_customers') }} c
  ON o.customer_id = c.customer_id
```
3. Or ensure dim_customers includes all referenced customers before fact_orders runs.

---

### D5: Custom Test Failure

**Custom Test File (tests/test_order_amount_positive.sql):**
```sql
-- tests/test_order_amount_positive.sql
SELECT *
FROM {{ ref('fact_orders') }}
WHERE order_amount <= 0  -- Find rows where this is true (should be empty)
```

**Error Message:**
```
Assertion failed. Found 5 rows with order_amount <= 0.
```

**What It Means:**
The custom test found rows that violate the assertion (rows with negative or zero amount).

**How to Read Custom Test SQL:**
- A passing test returns 0 rows.
- A failing test returns rows that VIOLATED the condition.
- The WHERE clause defines what "bad" data looks like.

**How to Fix the Root Cause:**
1. Diagnose which rows fail:
```sql
SELECT * FROM {{ ref('fact_orders') }}
WHERE order_amount <= 0;
```
2. Trace upstream to find where negative amounts come from.
3. Fix in the model (e.g., use ABS() or filter them out):
```sql
SELECT
  order_id,
  ABS(order_amount) AS order_amount
FROM {{ ref('raw_orders') }}
```


<hr style="border: 2px solid black;">

## Pattern E: Dependency & DAG Errors {#pattern-e}

### E1: Orphan Model (Not Referenced, Not in Selector)

**Issue:**
You have a model that is never used by any other model and is not explicitly selected during a run.

**Broken Scenario:**
```
models/staging/stg_products.sql  -- Created but never used
models/marts/fact_orders.sql     -- Only depends on stg_orders, not stg_products
```

**What It Means:**
This model is "dead code" in dbt. It won't be run unless explicitly selected with `dbt run --select stg_products`.

**Fix:**
1. If the model is needed, add a ref() to it from another model:
```sql
-- models/marts/fact_orders.sql
SELECT
  o.order_id,
  p.product_name
FROM {{ ref('stg_orders') }} o
JOIN {{ ref('stg_products') }} p  -- Add the dependency
  ON o.product_id = p.product_id
```
2. If the model is not needed, delete it.

---

### E2: Model Runs Before Its Dependency (Missing ref())

**Issue:**
A model attempts to use data from an upstream model, but doesn't use ref(). Instead, it hardcodes a table name.

**Broken Code:**
```sql
-- models/marts/fact_orders.sql
SELECT * FROM schema.stg_orders  -- Hardcoded table name!
```

**What It Means:**
dbt doesn't know that fact_orders depends on stg_orders, so it might run fact_orders before stg_orders is created. This causes the "table not found" error.

**Fix:**
Always use ref() to declare dependencies:
```sql
SELECT * FROM {{ ref('stg_orders') }}
```
Now dbt knows to run stg_orders first.

---

### E3: Running Models Out of Order (dbt run vs dbt build)

**Scenario:**
```bash
dbt run
# All models run, but some may have missing dependencies

dbt test
# Tests run on all models, including those that failed to build
```

**What It Means:**
- `dbt run` runs all models but ignores test failures.
- `dbt build` runs models → tests → stops on failure.
- `dbt build` is safer for CI/CD because it fails fast.

**Fix:**
Use `dbt build` instead of `dbt run` + `dbt test`:
```bash
dbt build --select fact_orders+
# Runs fact_orders and all downstream models
# Runs tests on fact_orders
# Stops if tests fail
```

---

### E4: Selector/Tag Filtering Excluding Needed Models

**Scenario:**
```bash
dbt run --select tag:daily
# Only runs models tagged with 'daily'
# But fact_orders (no tag) depends on stg_orders (tag: daily)
# Result: stg_orders not built, fact_orders fails
```

**What It Means:**
Selectors are powerful but can accidentally exclude dependencies. Always check the DAG before running.

**Fix:**
Use `dbt run --select +fact_orders` to run fact_orders and all upstream dependencies:
```bash
dbt run --select +fact_orders
# Runs all dependencies first, then fact_orders
```

---

### E5: dbt build vs dbt run Execution Order

**dbt run:**
1. Parse all models
2. Build DAG
3. Run all models in dependency order
4. Return list of failed models (but don't stop)

**dbt build:**
1. Parse all models
2. Build DAG
3. Run model → run tests for that model → run downstream models
4. Stop on first test failure (fail fast)

**Interview Tip:**
In production, use `dbt build` for safety. In development, use `dbt run` to debug faster (but always run tests before committing).


<hr style="border: 2px solid black;">

## Pattern F: Incremental Model Issues {#pattern-f}

### Decision Tree: Is My Incremental Model Broken?

<div class="fc">
  <div class="fc-node fc-start">Incremental Model Failing?</div>
  <div class="fc-arrow">▼</div>
  <div class="fc-branches fc-three">
    <div class="fc-branch">
      <div class="fc-label fc-tag">DUPLICATES</div>
      <div class="fc-node fc-warn"><strong>F1</strong> — Missing unique_key<br/>Rows inserted multiple times</div>
    </div>
    <div class="fc-branch">
      <div class="fc-label fc-tag">SCHEMA CHANGED</div>
      <div class="fc-node fc-warn"><strong>F2</strong> — New column in source<br/>Run --full-refresh</div>
    </div>
    <div class="fc-branch">
      <div class="fc-label fc-tag">LOGIC ERROR</div>
      <div class="fc-node fc-warn"><strong>F3</strong> — is_incremental() bug<br/>Full vs incremental behave differently</div>
    </div>
  </div>
</div>

---

### F1: Incremental Model Inserting Duplicates (Missing unique_key)

**Error:**
You run the incremental model multiple times and notice duplicate rows in the destination table.

**Broken Code (models/marts/fact_orders.sql):**
```sql
{{ config(
  materialized='incremental',
  unique_key='order_id'  -- Missing or incomplete!
) }}

SELECT
  order_id,
  customer_id,
  order_amount,
  CURRENT_TIMESTAMP AS loaded_at
FROM {{ source('raw', 'orders') }}
WHERE CAST(order_date AS DATE) = CAST(CURRENT_TIMESTAMP AS DATE)  -- Only today's orders
```

**What Happened:**
Without a `unique_key`, dbt appends all new rows to the table. If you run the model twice, you get duplicates.

**Fix:**
Define a `unique_key` (one or more columns that uniquely identify a row):
```sql
{{ config(
  materialized='incremental',
  unique_key='order_id'  -- dbt will upsert on this column
) }}

SELECT
  order_id,
  customer_id,
  order_amount,
  CURRENT_TIMESTAMP AS loaded_at
FROM {{ source('raw', 'orders') }}
WHERE CAST(order_date AS DATE) = CAST(CURRENT_TIMESTAMP AS DATE)
```

Now dbt will:
1. Check if order_id exists in the table.
2. If it does, UPDATE it.
3. If it doesn't, INSERT it.

---

### F2: Schema Change on Incremental Model (New Column Not Added)

**Error:**
You added a new column to the source data, but the incremental model table doesn't have it.

**Broken Scenario:**
```sql
-- Initial model run:
SELECT order_id, customer_id, order_amount FROM raw_orders

-- Later, raw_orders gets a new column: order_notes
-- But the incremental model still only selects the old 3 columns
-- Result: order_notes never gets added to the destination table
```

**Fix:**
Run a full refresh to rebuild the table with the new schema:
```bash
dbt run --select fact_orders --full-refresh
```

Or update the model to include the new column:
```sql
{{ config(
  materialized='incremental',
  unique_key='order_id',
  on_schema_change='fail'  -- Fail if schema doesn't match
) }}

SELECT
  order_id,
  customer_id,
  order_amount,
  order_notes  -- New column added
FROM {{ source('raw', 'orders') }}
```

---

### F3: is_incremental() Logic Error

**Error:**
The model behaves differently when run incrementally vs full refresh. Data is missing or incorrect.

**Broken Code (models/marts/fact_orders.sql):**
```sql
{{ config(
  materialized='incremental',
  unique_key='order_id'
) }}

SELECT
  order_id,
  customer_id,
  order_amount
FROM {{ source('raw', 'orders') }}
{% if execute and execute %}
  WHERE order_date >= (SELECT MAX(order_date) FROM {{ this }} - INTERVAL 7 DAY)  -- Bug: MAX could be NULL on first run!
{% endif %}
```

**What Happened:**
On the first incremental run, the destination table doesn't exist, so `MAX(order_date)` returns NULL. The WHERE clause becomes `WHERE order_date >= NULL`, which filters everything out.

**Fix:**
Use dbt's built-in `is_incremental()` macro:
```sql
{{ config(
  materialized='incremental',
  unique_key='order_id'
) }}

SELECT
  order_id,
  customer_id,
  order_amount
FROM {{ source('raw', 'orders') }}
{% if is_incremental() %}
  WHERE order_date >= (SELECT MAX(order_date) FROM {{ this }})
{% endif %}
```

`is_incremental()` returns True only if the table exists and `dbt run --incremental` is used. On first run, it's False, so all data is loaded.

---

### F4: When to Use --full-refresh

Use `dbt run --full-refresh` when:
1. Schema changed (new column added or removed)
2. unique_key changed
3. Incremental logic is broken and you need to rebuild from scratch
4. Historical data needs to be reloaded

Example:
```bash
# Run all incremental models with full refresh
dbt run --full-refresh

# Run specific model with full refresh
dbt run --select fact_orders --full-refresh
```


<hr style="border: 2px solid black;">

## Pattern G: Source Freshness & Environment Issues {#pattern-g}

### G1: Source Freshness Failure

**Error Message:**
```
Source 'raw.orders' has freshness warning: last loaded 2 days ago (threshold: 24 hours)
```

**Source Definition (models/sources.yml):**
```yaml
version: 2
sources:
  - name: raw
    tables:
      - name: orders
        freshness:
          warn_after: {count: 24, period: hour}
        loaded_at_field: loaded_timestamp
```

**What It Means:**
The source table hasn't been updated in more than 24 hours. The data might be stale.

**Fix:**
1. Check if the upstream pipeline (that loads raw.orders) is running.
2. Adjust the freshness threshold if 24 hours is too strict:
```yaml
- name: orders
  freshness:
    warn_after: {count: 48, period: hour}  # Allow 48 hours
```
3. Or remove the freshness check if not needed.

---

### G2: Environment Variable Not Set in dbt Cloud

**Error Message:**
```
ERROR: Undefined variable 'my_env_var' in dbt Cloud job
```

**Broken Code (models/fact_orders.sql):**
```sql
SELECT * FROM {{ source('raw', 'orders') }}
WHERE load_date > '{{ env_var("START_DATE") }}'  -- Environment variable not set!
```

**Fix:**
1. Set the variable in dbt Cloud job settings:
   - Go to dbt Cloud → Job Settings → Environment Variables
   - Add: START_DATE = 2024-01-01
2. Or set in dbt_project.yml:
```yaml
vars:
  START_DATE: 2024-01-01
```
3. Use with default value for safety:
```sql
WHERE load_date > '{{ env_var("START_DATE", "2024-01-01") }}'
```

---

### G3: Profile/Target Mismatch (dev vs prod)

**Error Message:**
```
ERROR: Target 'prod' not found in profiles.yml
```

**Broken ~/.dbt/profiles.yml:**
```yaml
my_project:
  target: prod  # Set to prod
  outputs:
    dev:  # Only 'dev' defined, 'prod' missing!
      type: postgres
      host: localhost
```

**Fix:**
1. Define the 'prod' target:
```yaml
my_project:
  target: dev  # Or use 'prod' if it's defined below
  outputs:
    dev:
      type: postgres
      host: localhost
    prod:
      type: postgres
      host: prod-warehouse.com
```
2. Or use `dbt run --target dev` to override.

**In dbt Cloud:** Profiles are managed by dbt Cloud, not locally. Set the target in Job Settings → Environment → Target name.

---

### G4: Package Dependency Conflicts (packages.yml)

**Error Message:**
```
ERROR: Package version conflict: dbt_utils requires dbt >= 1.3.0, but you have dbt 1.2.0
```

**Broken packages.yml:**
```yaml
packages:
  - package: dbt-labs/dbt_utils
    version: 1.0.0  # Requires dbt >= 1.3.0
```

**Fix:**
1. Upgrade dbt:
```bash
pip install --upgrade dbt-core
```
2. Or downgrade the package:
```yaml
packages:
  - package: dbt-labs/dbt_utils
    version: 0.9.0  # Compatible with dbt 1.2.0
```
3. Or remove the package if not needed.


<hr style="border: 3px solid black;">

# Section 4: Code Review Checklist for dbt {#section-4}

When reviewing dbt code, use this three-pass checklist. Each pass focuses on a different aspect of code quality.

## Pass 1: Structure

Check the organization, naming, and dependency declarations.

| Check | What to Look For | Red Flag |
|---|---|---|
| **Naming Convention** | Model names are lowercase, snake_case, descriptive (e.g., `stg_orders`, `fct_revenue`, `dim_customers`) | Names like `my_model`, `temp`, `table1`, `stg_raw_*` (too verbose) |
| **Folder Organization** | Staging models in `staging/`, marts in `marts/`, intermediate in `intermediate/` | Models scattered randomly, no clear folder structure |
| **ref() Usage** | All dependencies use `ref('model_name')` to reference other dbt models | Hardcoded table names like `schema.table` or `"raw"."orders"` |
| **source() Usage** | Raw data referenced via `source('source_name', 'table_name')` from schema.yml | Hardcoded `SELECT * FROM raw.my_table` without source definition |
| **No Hardcoded Schema** | Model doesn't reference hardcoded schema names (except in comments) | `FROM my_schema.orders` instead of `FROM {{ source(...) }}` |
| **SELECT * Avoided** | Model explicitly lists columns, not `SELECT *` | `SELECT * FROM upstream_model` in a production model |
| **CTEs Organized** | Common Table Expressions (CTEs) are named clearly: `with renamed_columns as`, `with filtered_orders as` | CTEs named `a`, `b`, `c` or `temp`, `temp2` |

---

## Pass 2: Logic

Check the SQL logic, transformations, and model configurations.

| Check | What to Look For | Red Flag |
|---|---|---|
| **Correct Materialization** | Materialization matches use case: `view` (cheap, real-time), `table` (expensive, snapshot), `incremental` (append/upsert) | Using `table` for a simple lookup or `view` for a 10GB denormalized fact table |
| **Join Type Correct** | INNER JOINs when cardinality is 1:1 or M:1, LEFT JOINs only when necessary to preserve nulls | INNER JOIN that should be LEFT (losing data) or LEFT JOIN that should be INNER (introducing nulls) |
| **Grain of Model** | The grain (uniqueness level) is correct and documented (e.g., "one row per order_id") | Model says it's one row per order but has duplicate order_ids |
| **Aggregation Correct** | GROUP BY includes all non-aggregated columns, no implicit conversions | Selecting columns not in GROUP BY, or SUM(column) without GROUP BY |
| **Window Functions** | PARTITION BY and ORDER BY clauses are appropriate for the metric | RANK() without ORDER BY or PARTITION BY all rows (useless) |
| **Incremental Logic** | For incremental models: unique_key is defined, is_incremental() is used correctly | Incremental model without unique_key or is_incremental() logic |
| **Date Filtering** | Filtering by date is clear and efficient (not CAST(date_col AS DATE) = CAST(CURRENT_TIMESTAMP AS DATE)) | Expensive date conversions in WHERE clause |
| **Case Statements** | CASE WHEN logic is clear and handles NULL values | Incomplete CASE statements or CASE WHEN column IN (select...) |
| **No Business Logic in Staging** | Staging models only rename/type-cast; mart models handle business logic | stg_orders has revenue calculations or business rules |

---

## Pass 3: Quality

Check for tests, documentation, and data quality safeguards.

| Check | What to Look For | Red Flag |
|---|---|---|
| **Primary Key Test** | Model with a primary key column has a `unique` and `not_null` test | No test on `id` or `order_id` (primary key has no tests) |
| **Foreign Key Test** | Models that reference other models have `relationships` test | No test checking that customer_id in fact_orders exists in dim_customers |
| **Critical Column Tests** | Important business columns have `not_null` or `accepted_values` tests | Revenue column has no `not_null` test |
| **Model Documentation** | Model has a description in schema.yml explaining its purpose | Missing `description:` field in model definition |
| **Column Documentation** | Important columns documented in schema.yml with business meaning | Columns like `amount`, `status`, `date` have no descriptions |
| **Source Freshness Defined** | Sources have `freshness:` thresholds if they're time-sensitive | Data source marked as "last updated 3 days ago" with no alert |
| **Pre-Hooks/Post-Hooks** | Hooks are minimal and purposeful (e.g., grant permissions, log runs) | Heavy transformation logic in pre-hooks instead of models |
| **Macro Usage** | Custom macros are parameterized and reusable | Macros with hardcoded values or complex nested logic |
| **No Warnings** | dbt compile and dbt run show no warnings | `Column is unused` warnings or `Unused test` warnings |
| **No Sensitive Data** | No passwords, API keys, or personally identifiable info (PII) in models | Password hardcoded in model or PII not masked in development |

---

## dbt Best Practices: Staging → Intermediate → Mart Pattern

### Staging Models (stg_*)
- Ingest raw tables from sources
- Rename columns to be consistent (snake_case)
- Cast data types (strings to dates, etc.)
- Remove junk rows (filter out nulls if needed)
- **NO** business logic, calculations, or aggregations
- **NO** JOINs (unless renaming columns from multiple tables)
- Usually materialized as `view` (cheap)

### Intermediate Models (int_*)
- Build on staging models with logical transformations
- Add calculated columns (e.g., `order_total_usd = quantity * unit_price`)
- Join related tables (e.g., orders + customers)
- Aggregate when necessary (e.g., orders → order_summary)
- Still focused on one business concept
- Usually materialized as `view` unless expensive to rebuild

### Mart Models (fct_*, dim_*)
- Serve as the source of truth for BI and reporting
- `fct_*` = fact table (granular events/transactions, many rows)
- `dim_*` = dimension table (attributes, reference data, few rows)
- Apply all business logic and metrics
- Denormalize for performance (wider tables OK)
- Usually materialized as `table` (snapshot) or `incremental` (performance)
- Have comprehensive tests and documentation


<hr style="border: 3px solid black;">

# Section 5: Debugging Workflow in dbt Cloud {#section-5}

When you encounter a dbt error, follow this 5-step workflow to systematically debug and fix it.

<div class="fc">
  <div class="fc-node fc-start">dbt Run Failed</div>
  <div class="fc-arrow">▼</div>
  <div class="fc-branches fc-three">
    <div class="fc-branch">
      <div class="fc-label fc-tag">STEP 1</div>
      <div class="fc-node fc-action"><strong>READ</strong><br/>Error Panel — which step failed?<br/>(Parse / Compile / Run / Test)</div>
    </div>
    <div class="fc-branch">
      <div class="fc-label fc-tag">STEP 2</div>
      <div class="fc-node fc-action"><strong>IDENTIFY</strong><br/>Which model in the DAG?<br/>Use Lineage View</div>
    </div>
    <div class="fc-branch">
      <div class="fc-label fc-tag">STEP 3</div>
      <div class="fc-node fc-action"><strong>ISOLATE</strong><br/>dbt compile or Preview<br/>See the compiled SQL</div>
    </div>
  </div>
  <div class="fc-arrow">▼</div>
  <div class="fc-branches">
    <div class="fc-branch">
      <div class="fc-label fc-tag">STEP 4</div>
      <div class="fc-node fc-action"><strong>FIX</strong><br/>Root cause: YAML / Jinja / SQL / Data?<br/>Apply pattern from library</div>
    </div>
    <div class="fc-branch">
      <div class="fc-label fc-tag">STEP 5</div>
      <div class="fc-node fc-end"><strong>VERIFY</strong><br/><code>dbt build --select model+</code><br/>Run model + downstream tests</div>
    </div>
  </div>
</div>

---

## Step 1: READ the Error Panel

Open the dbt Cloud **Logs** tab and read the error carefully.

**Ask yourself:**
1. At which step did it fail?
   - `Parsing` (YAML error) → Pattern A
   - `Compiling` (Jinja error) → Pattern B
   - `Running SQL` (database error) → Pattern C
   - `Running test` (test failure) → Pattern D

2. What is the exact error message?
   - Copy the error message to match it against the pattern library.

3. Which file failed?
   - Look for "in model" or "in file" to identify the model path.

---

## Step 2: IDENTIFY the Model

Determine which model(s) in the DAG are affected.

**dbt Cloud tools:**
- **Lineage View**: Shows the DAG visually. Click on the failed model to see dependencies.
- **Run Details**: Lists all models executed and their status (success, failure, skipped).
- **Logs Panel**: Shows which model failed and the error message.

**Ask yourself:**
1. Is the failing model a staging model, intermediate, or mart?
2. What models does it depend on (upstream)?
3. What models depend on it (downstream)?
4. Could the error be in an upstream model instead?

---

## Step 3: ISOLATE Using dbt compile or Preview

See the actual SQL that dbt is trying to execute.

**dbt Cloud Tools:**
- **Compile Button**: Compiles all models without executing them. Check the Logs tab for compiled SQL.
- **Preview Button**: Runs a quick preview of the model SQL. Shows first few rows if no errors.
- **IDE**: Open the model file and look for syntax errors (red squiggly lines).

**What to look for:**
- If compilation fails with a Jinja error, check for missing closing `}}`, undefined ref(), or incorrect macro calls.
- If SQL preview fails, look for:
  - Column names that don't exist in upstream models
  - Syntax errors (missing commas, wrong keywords)
  - Ambiguous column references in JOINs

**Example:**
```
dbt compile (in dbt Cloud IDE)
# or
dbt compile --select my_model
# Check logs/target/compiled/project/models/my_model.sql
```

---

## Step 4: FIX the Root Cause

Once you've identified the problem, apply the fix from the pattern library.

**Fix types:**
- **YAML error** (Pattern A): Fix indentation, add missing fields, check for duplicates
- **Jinja error** (Pattern B): Fix ref/source calls, Jinja syntax, macro arguments
- **SQL error** (Pattern C): Fix column names, types, JOINs, WHERE clauses
- **Test failure** (Pattern D): Trace upstream to find the root data issue
- **DAG error** (Pattern E): Add missing ref() or fix selector logic
- **Incremental error** (Pattern F): Add unique_key or fix is_incremental() logic
- **Environment error** (Pattern G): Set env var, check freshness, fix profile

**Edit the model:**
1. In dbt Cloud IDE, open the model file.
2. Make the fix.
3. Save the file (Ctrl+S or Cmd+S).

---

## Step 5: VERIFY with dbt build --select

Test the fix by running the model and its tests.

**Command:**
```bash
dbt build --select my_model+
```

This runs:
1. All upstream dependencies (the `+` prefix means dependencies)
2. The model itself
3. All tests on the model
4. All downstream models

**Expected result:**
- Green checkmarks on all models and tests.
- If any test fails, trace upstream and fix the data issue (Pattern D).

**If still failing:**
1. Revisit Step 1: Did you read the error correctly?
2. Revisit Step 3: Is the compiled SQL what you expected?
3. Check the warehouse query history for more details.

---

## dbt Cloud-Specific Debugging Tools

| Tool | What It Does | When to Use |
|---|---|---|
| **Logs Tab** | Shows execution logs, error messages, compilation output | Always start here to read the error |
| **Compile Button** | Compiles models without executing SQL | Check for Jinja/ref errors before running |
| **Preview Button** | Runs a quick preview of the first rows | Verify the SQL logic and column names |
| **Lineage View** | Visual DAG showing model dependencies | Understand the data flow and spot missing refs |
| **IDE** | Built-in editor with syntax highlighting | Edit models and see real-time Jinja errors |
| **Diff Viewer** | Shows changes between commits | Review PR changes before merge |
| **Run Details** | Lists all models, tests, and their status | Track which models failed and which passed |
| **Warehouse Query History** | Direct view of SQL executed in the data warehouse | See detailed error messages from the warehouse |


<hr style="border: 3px solid black;">

# Section 6: Common Interview Traps {#section-6}

During a technical interview, the interviewer may intentionally plant bugs in the code to test your debugging skills. Here are the most common traps and what the interviewer is looking for when they plant them.

| Trap | What the Interviewer Planted | What They Want You to Say |
|---|---|---|
| **Hardcoded table name instead of ref()** | `FROM schema.orders` instead of `FROM {{ ref('stg_orders') }}` | "This breaks the dependency chain. dbt doesn't know that my_model depends on stg_orders, so it might run before stg_orders is created. I'd change it to ref()." |
| **Missing test on primary key** | Model has `id` column but no `unique` or `not_null` test | "Every primary key should have both a unique and not_null test to ensure data quality. I'd add those tests to schema.yml." |
| **SELECT * in production model** | Mart model uses `SELECT * FROM upstream` | "SELECT * is dangerous because if upstream adds or removes columns, it breaks downstream consumers. I'd explicitly list the columns I need." |
| **Wrong materialization** | Materialization is `view` for a large denormalized fact table | "This should be materialized as a table because views are recomputed on every query. For a large aggregate used by many BI dashboards, a table is more efficient." |
| **No source freshness check** | Source table isn't checked for staleness | "If this source is time-sensitive, I'd add a freshness check to schema.yml. If it hasn't been updated in 24 hours, dbt should warn us." |
| **Macro with hardcoded logic** | Macro has hardcoded column names or values | "This macro isn't reusable. I'd parameterize it to accept column names as arguments, like `{% macro clean_column(column_name) %}`." |
| **Incremental model without unique_key** | Incremental model has `materialized='incremental'` but no `unique_key` | "This model will insert duplicates every time it runs. I need to add `unique_key='order_id'` to upsert instead of append." |
| **Staging model with business logic** | stg_orders calculates revenue or applies discount rules | "Staging models should only rename columns and cast types. Business logic belongs in mart models, not staging." |
| **JOIN without specifying type** | `JOIN customers c ON ...` (implicit INNER JOIN) | "I'd explicitly use INNER JOIN or LEFT JOIN depending on whether I want to preserve unmatched rows. Implicit JOINs are risky." |
| **Case statement without NULL handling** | `CASE WHEN status = 'active' THEN 1 ELSE 0 END` (doesn't handle NULL) | "This treats NULL as 0, which might be wrong. I'd add an explicit WHEN status IS NULL THEN NULL to preserve unknown values." |
| **Circular reference** | Model A refs B, Model B refs A | "This creates a circular dependency that dbt can't resolve. I'd refactor to remove one of the refs, maybe creating an intermediate model." |
| **Missing incremental logic** | Incremental model without `is_incremental()` check | "On the first run, the table doesn't exist, so MAX(load_date) is NULL and no data loads. I'd use `{% if is_incremental() %}` to only filter on first run." |
| **Test on incorrect column** | `unique` test on a column that should have duplicates | "This test is wrong. If an order can have multiple line items, order_id will be a duplicate. I'd remove this test or add it to (order_id, line_item_id)." |
| **Wrong ref() name (typo)** | `ref('custormers')` instead of `ref('customers')` | "dbt can't find a model named 'custormers'. I'd check the exact filename and fix the typo in the ref() call." |
| **No documentation** | Model has no description in schema.yml | "Without documentation, downstream users don't know what this model is for or when to use it. I'd add descriptions to the model and key columns." |
